# Project 2 — Bank Marketing Analysis (2025)

**Goal:** Practice Python + SQL and turn campaign data into business recommendations.

**Dataset:** Kaggle Playground Series S5E8 — Binary Classification with a Bank Dataset. This is a synthetic dataset based on the original Bank Marketing data; it is not a record of actual campaigns that occurred in 2025.

## How to use
1. Download `train.csv` from the [Kaggle competition Data tab](https://www.kaggle.com/competitions/playground-series-s5e8/data).
2. Run the cells from top to bottom.
3. When prompted, upload `train.csv`.
4. Review the data-quality and SQL outputs before writing conclusions.

**Important:** Do not claim causal effects from descriptive comparisons. The target `y` is the subscription outcome.


## 1. Load the dataset
Upload `train.csv` when the file picker appears.

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
import pandas as pd

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("No CSV uploaded. Please upload train.csv from the Kaggle dataset.")

preferred = [name for name in csv_files if name.lower() == "train.csv"]
file_name = preferred[0] if preferred else csv_files[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"Loaded: {file_name}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())


## 2. Understand the data
Check column names, types, missing values, duplicates, and target balance.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values (top 15):")
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

if "y" not in df.columns:
    raise KeyError("Expected target column 'y' was not found. Check that you uploaded the competition train.csv.")

print("\nTarget counts:")
display(df["y"].value_counts(dropna=False).to_frame("count"))

print("\nTarget share (%):")
display((df["y"].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame("percent"))


## 3. Basic Python EDA
First inspect the available categories. These summaries are descriptive, not causal.

In [ ]:
candidate_columns = ["job", "education", "marital", "contact", "month", "campaign", "previous", "poutcome", "age"]
available_columns = [col for col in candidate_columns if col in df.columns]
print("Available candidate columns:", available_columns)

def segment_summary(data, column, target="y"):
    summary = (
        data.groupby(column, dropna=False)[target]
        .agg(contacts="size", subscriptions=lambda s: (s == "yes").sum())
        .reset_index()
    )
    summary["subscription_rate_pct"] = (
        summary["subscriptions"] / summary["contacts"] * 100
    ).round(2)
    return summary.sort_values(["subscription_rate_pct", "contacts"], ascending=[False, False])

for col in ["job", "contact", "month", "poutcome"]:
    if col in df.columns:
        print(f"\nSubscription rate by {col}:")
        display(segment_summary(df, col))


## 4. Visualize subscription rate
This plots subscription rates, not raw counts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if "contact" in df.columns:
    plot_col = "contact"
elif available_columns:
    plot_col = available_columns[0]
else:
    raise ValueError("No candidate segment columns are available.")

plot_data = segment_summary(df, plot_col).sort_values("subscription_rate_pct", ascending=True)
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_data, x="subscription_rate_pct", y=plot_col)
plt.title(f"Term-deposit subscription rate by {plot_col}")
plt.xlabel("Subscription rate (%)")
plt.ylabel(plot_col)
plt.tight_layout()
plt.show()


## 5. Practice SQL with SQLite
The DataFrame is loaded into an in-memory SQLite table called `bank_marketing`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
df.to_sql("bank_marketing", conn, if_exists="replace", index=False)

def run_sql(query):
    return pd.read_sql_query(query, conn)

display(run_sql("""
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
    ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS subscription_rate_pct
FROM bank_marketing;
"""))


### SQL exercise 1 — Compare job segments
Try changing the sort order or minimum group size.

In [ ]:
display(run_sql("""
SELECT
    job,
    COUNT(*) AS contacts,
    SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
    ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS subscription_rate_pct
FROM bank_marketing
GROUP BY job
HAVING COUNT(*) >= 100
ORDER BY subscription_rate_pct DESC;
"""))


### SQL exercise 2 — Compare contact channel
Check both rate and volume.

In [ ]:
if "contact" in df.columns:
    display(run_sql("""
    SELECT
        contact,
        COUNT(*) AS contacts,
        SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
        ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS subscription_rate_pct
    FROM bank_marketing
    GROUP BY contact
    ORDER BY subscription_rate_pct DESC;
    """))
else:
    print("No 'contact' column in this version. Inspect the columns and adapt the query.")


## 6. Business interpretation worksheet

Use actual outputs to answer:
1. Which segment has the highest subscription rate, and how many contacts are in it?
2. Does the highest-rate segment also have meaningful volume?
3. What small, measurable campaign test would you propose?
4. What other factors could explain the difference?
5. How would you evaluate the test (subscription rate, number of subscriptions, cost per subscription, contact volume)?

Avoid saying a segment *causes* more subscriptions from descriptive results alone. Prefer: “This segment was associated with a higher subscription rate in this dataset.”


## 7. Export a data-quality report (optional)

In [ ]:
quality_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": [int(df[col].isna().sum()) for col in df.columns],
    "missing_pct": [round(float(df[col].isna().mean() * 100), 2) for col in df.columns],
})
quality_report.to_csv("data_quality_report.csv", index=False)
print("Saved data_quality_report.csv")


## Next steps
- Add SQL queries for education, month, and campaign frequency.
- Inspect categorical values such as `unknown`.
- Write a README with methods, validated findings, limitations, and recommendations.
- Save the notebook to GitHub after checking it runs top to bottom.

Dataset source: [Kaggle Playground Series S5E8](https://www.kaggle.com/competitions/playground-series-s5e8/data)
